In [0]:
import requests
import json
import time
import pandas as pd
from pyspark.sql import functions as F

In [0]:
import requests
import json
import time
import pandas as pd
from pyspark.sql import functions as F

TOMTOM_API_KEY = "STF12pYdXlKI5XscNENrqftfRgpHmOZt"  # Замените на dbutils.secrets.get()
CATALOG = "dbr_dev"
SCHEMA = "artemzharkov10_bronze"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.bronze_tomtom_raw"

url = "https://api.tomtom.com/traffic/services/5/incidentDetails"

min_lon, min_lat, max_lon, max_lat = 14.12, 49.00, 24.15, 54.84
step_lon, step_lat = 1.2, 0.8

raw_records = []
lon = min_lon

while lon < max_lon:
    lat = min_lat
    while lat < max_lat:
        current_max_lon = min(lon + step_lon, max_lon)
        current_max_lat = min(lat + step_lat, max_lat)
        current_bbox = f"{lon},{lat},{current_max_lon},{current_max_lat}"
        
        params = {
            "key": TOMTOM_API_KEY,
            "bbox": current_bbox,
            "language": "pl-PL",
            "fields": "{incidents{geometry{coordinates},properties{id,iconCategory,magnitudeOfDelay,startTime,events{description}}}}"
        }
        
        response = requests.get(url, params=params, timeout=20)
        
        if response.status_code == 200:
            data = response.json()
            incidents = data.get("incidents", [])
            
            # Сохраняем каждый инцидент из ответа API в чистом виде без фильтрации
            for item in incidents:
                json_string = json.dumps(item, ensure_ascii=False)
                raw_records.append({"raw_json": json_string})
        else:
            print(f"Ошибка сектора {current_bbox}: {response.status_code}")
            
        lat += step_lat
        time.sleep(0.2)
        
    lon += step_lon

if raw_records:
    pdf = pd.DataFrame(raw_records)
    
    # Дедупликация идентичных сырых записей
    pdf.drop_duplicates(subset=["raw_json"], inplace=True)
    
    df_raw = spark.createDataFrame(pdf)
    df_raw = df_raw.withColumn("ingest_timestamp", F.current_timestamp())
    
    (
        df_raw.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )
    print(f"Записано {len(pdf)} сырых объектов в таблицу {TARGET_TABLE}.")
else:
    print("API не вернул данных.")

In [0]:
df = spark.table(TARGET_TABLE)
display(df)